# Project 2 Milestone 1: Feature-Space Design\n\nThis notebook starts Project 2 of the Gaia–LAMOST Galactic Archaeology research sequence.\n\nProject 2 focuses on machine-learning assisted stellar population clustering using the larger Gaia–LAMOST velocity-feature table produced in Project 1.\n\nThe goal of this milestone is not to claim any astrophysical discovery. Instead, it defines the input dataset, inspects available features, and prepares several feature-space groups for later clustering experiments.

## Research Question\n\nCan unsupervised machine learning recover interpretable stellar population structure from Gaia–LAMOST chemo-kinematic features?

## Primary Input Dataset\n\nThe primary Project 2 input table is:\n\n`../data/processed/gaia_lamost_larger_velocity_features.csv`\n\nThis table contains Gaia astrometry and photometry, LAMOST spectroscopic parameters, derived kinematic quantities, Galactic coordinates, and Galactocentric velocity features.

In [ ]:
from pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\npd.set_option('display.max_columns', 100)\npd.set_option('display.width', 160)

In [ ]:
DATA_PATH = Path('../data/processed/gaia_lamost_larger_velocity_features.csv')\n\ndf = pd.read_csv(DATA_PATH)\ndf.shape

In [ ]:
df.head()

## Available Columns

In [ ]:
for col in df.columns:\n    print(col)

## Candidate Interpretation Columns\n\nThe following Project 1 columns are useful for interpretation, but should not be used as input labels for unsupervised clustering.

In [ ]:
interpretation_cols = [\n    'metallicity_group',\n    'high_vtan_candidate',\n    'metal_poor_candidate',\n    'chemo_kinematic_candidate',\n]\n\nfor col in interpretation_cols:\n    if col in df.columns:\n        print(f'\\n{col}')\n        print(df[col].value_counts(dropna=False))

## Feature-Space Definitions\n\nSeveral feature spaces are defined for later clustering experiments. These groups separate photometric, chemical, kinematic, and combined chemo-kinematic information.

In [ ]:
feature_spaces = {\n    'photometric_astrometric': [\n        'bp_rp',\n        'absolute_g_mag',\n        'distance_pc',\n        'pm_total',\n        'reduced_pm_g',\n    ],\n    'chemical_stellar': [\n        'feh',\n        'teff',\n        'logg',\n    ],\n    'local_kinematic': [\n        'tangential_velocity_kms',\n        'rv',\n    ],\n    'galactocentric_velocity': [\n        'galcen_vx_kms',\n        'galcen_vy_kms',\n        'galcen_vz_kms',\n        'galcen_vtot_kms',\n    ],\n    'combined_chemo_kinematic': [\n        'feh',\n        'bp_rp',\n        'absolute_g_mag',\n        'tangential_velocity_kms',\n        'rv',\n        'galcen_vx_kms',\n        'galcen_vy_kms',\n        'galcen_vz_kms',\n        'galcen_vtot_kms',\n    ],\n}\n\nfeature_spaces

## Feature Availability and Missingness\n\nBefore clustering, each feature space should be checked for missing values and valid sample size.

In [ ]:
summary_rows = []\n\nfor name, cols in feature_spaces.items():\n    missing_cols = [col for col in cols if col not in df.columns]\n    available_cols = [col for col in cols if col in df.columns]\n    complete_rows = df[available_cols].dropna().shape[0] if available_cols else 0\n    summary_rows.append({\n        'feature_space': name,\n        'n_features_defined': len(cols),\n        'n_features_available': len(available_cols),\n        'missing_columns': ', '.join(missing_cols),\n        'complete_rows': complete_rows,\n    })\n\nfeature_space_summary = pd.DataFrame(summary_rows)\nfeature_space_summary

## Basic Numeric Summary

In [ ]:
numeric_cols = sorted(set(sum(feature_spaces.values(), [])))\nnumeric_cols = [col for col in numeric_cols if col in df.columns]\n\ndf[numeric_cols].describe().T

## Save Project 2 Feature-Space Summary\n\nThis table records which feature spaces are ready for later clustering experiments.

In [ ]:
OUTPUT_PATH = Path('../data/processed/project2_feature_space_summary.csv')\nfeature_space_summary.to_csv(OUTPUT_PATH, index=False)\nOUTPUT_PATH

## Milestone 1 Notes\n\n- Project 2 will use unsupervised methods only at this stage.\n- Project 1 candidate flags will be used only for post-hoc interpretation.\n- Clustering results should be interpreted as feature-space structures, not confirmed Galactic substructures.\n- Later milestones will compare PCA, UMAP, DBSCAN/HDBSCAN, and Gaussian Mixture Models.